In [0]:
%pip install --upgrade "trafilatura" "mlflow[databricks]" "langchain>=1.2" "langgraph>=1.0" "grandalf" -q
%restart_python

In [0]:
import trafilatura
import os
import mlflow
from langchain_openai.chat_models.azure import AzureChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain.agents import create_agent
from langchain.tools import tool
from langchain.messages import SystemMessage, HumanMessage, ToolMessage, AIMessage
from langgraph.graph import MessagesState
from langgraph.graph import StateGraph, START, END
from typing_extensions import Literal
from IPython.display import Image, display


dbutils.widgets.text("azure_openai_chat_deployment_name", "gpt-4o")
dbutils.widgets.text("azure_openai_api_version", "2024-10-21")
dbutils.widgets.text("model_version", "2024-11-20")

os.environ["AZURE_OPENAI_CHAT_DEPLOYMENT_NAME"] = dbutils.widgets.get("azure_openai_chat_deployment_name")
os.environ["AZURE_OPENAI_API_VERSION"] = dbutils.widgets.get("azure_openai_api_version")
os.environ["MODEL_VERSION"] = dbutils.widgets.get("model_version")
AZURE_OPENAI_ENDPOINT = dbutils.secrets.get(scope="demo-scope", key="azure_openai_endpoint")
AZURE_OPENAI_API_KEY = dbutils.secrets.get(scope="demo-scope", key="azure_openai_api_key")

# set the env vars
os.environ["AZURE_OPENAI_ENDPOINT"] = AZURE_OPENAI_ENDPOINT
os.environ["AZURE_OPENAI_API_KEY"] = AZURE_OPENAI_API_KEY

mlflow.langchain.autolog()

# For more control over the model configuration, initialize a model instance directly using the provider package. 
llm_client = AzureChatOpenAI(
    azure_deployment=os.environ.get("AZURE_OPENAI_CHAT_DEPLOYMENT_NAME"),
    api_version=os.environ.get("AZURE_OPENAI_API_VERSION"),
    model_version=os.environ.get("MODEL_VERSION"),
)


# alternatively, use the Databricks Chat model on the workspace
# from databricks.sdk import WorkspaceClient

# SECRET_SCOPE = "demo-scope"
# os.environ["DATABRICKS_HOST"] = "https://dbc-564fb500-5a75.cloud.databricks.com/"
# os.environ["DATABRICKS_MODEL"] = "databricks-gpt-oss-20b"
# os.environ["TEMPERATURE"] = "0"
# os.environ["DATABRICKS_TOKEN"] = dbutils.secrets.get(scope=SECRET_SCOPE, key="gpt_oss_20b_databricks_token")
# workspace_client = WorkspaceClient(
#     host=os.environ.get("DATABRICKS_HOST"), token=os.environ.get("DATABRICKS_TOKEN")
# )
# llm_client = ChatDatabricks(
#     model=os.environ.get("DATABRICKS_MODEL"),
#     temperature=int(os.environ.get("TEMPERATURE")),
#     workspace_client=workspace_client,
# )


## Testing the web scraping tool

In [0]:
url = "https://www.canada.ca/en/immigration-refugees-citizenship/services/immigrate-canada/express-entry.html"
url = "https://www.ielts.ca/take-ielts/ielts-for-canadian-immigration/ielts-to-clb/"
downloaded = trafilatura.fetch_url(url)
content = trafilatura.extract(
    downloaded,
    output_format="markdown",
    include_links=True,
    include_tables=True,
    include_images=False,
)
print(content)

## Static Tools

In [0]:
def fetch_link_content(url:str) -> str:
    downloaded = trafilatura.fetch_url(url)
    content = trafilatura.extract(
        downloaded,
        output_format="markdown",
        include_links=True,
        include_tables=True,
        include_images=False,
    )
    return content

@tool("fetch_express_entry_intro_tool")
def fetch_express_entry_intro_tool(input: str = ""):
    """
    Scrapes the IRCC website for the latest introduction on the Express Entry system.
    """
    return fetch_link_content(url="https://www.canada.ca/en/immigration-refugees-citizenship/services/immigrate-canada/express-entry.html")

@tool("fetch_express_entry_documents_tool")
def fetch_express_entry_documents_tool(input: str = ""):
    """
    Scrapes the IRCC website for the latest list of documents to create a profile Express Entry system.
    """
    return fetch_link_content(url="https://www.canada.ca/en/immigration-refugees-citizenship/services/immigrate-canada/express-entry/documents.html")

@tool("fetch_language_test_document_tool")
def fetch_language_test_document_tool(input: str = ""):
    """
    Scrapes the IRCC website for:
    - How to use the language test result in your Express Entry profile
    - List of approved language tests
    - Required language level for English and French in 3 programs under Express Entry: 
        - Federal Skilled Worker Program (FSWP)
        - Federal Skilled Trades Program (FSTP)
        - Canadian Experience Class (CEC)
    - Validity of the language test result
    """
    return fetch_link_content(url="https://www.canada.ca/en/immigration-refugees-citizenship/services/immigrate-canada/express-entry/documents/language-test.html")

@tool("direct_to_clb_ielts_converter_tool")
def direct_to_clb_ielts_converter_tool(input: str = ""):
    """
    Direct users to the external website for CLB-IELTS conversion. This tool is not maintained internally.
    """
    message = """
    To convert your IELTS score to CLB or vice-versa, please visit the following website:
    https://www.ielts.ca/take-ielts/ielts-for-canadian-immigration/ielts-to-clb/
    """
    return message

tools = [
    fetch_express_entry_intro_tool,
    fetch_express_entry_documents_tool,
    fetch_language_test_document_tool,
    direct_to_clb_ielts_converter_tool]

tools_by_name = {tool.name: tool for tool in tools}

In [0]:
tool_message = fetch_language_test_document_tool.invoke("")

## Agent with Static Tools

In [0]:
SYSTEM_PROMPT = """
You are a helpful, patient and professional immigration agent that helps users answers questions on Canada Express Entry immigration system.
You are equiped with a list of tools to query the trusted information, based on which your answers must be.
However, as the content retrieved may contain more information than what is needed for the question, pick out only the necessary information to provide the answer.
If there is no tool to answer the user's question, politely decline to answer it.
"""
agent = create_agent(model=llm_client, tools=tools, system_prompt=SYSTEM_PROMPT)

In [0]:
type(agent)
display(Image(agent.get_graph(xray=True).draw_mermaid_png()))

## Agent Invocation

In [0]:
query = "tell me more about the express entry system and its function?"
result = agent.invoke(
    {"messages": [{"role": "user", "content": query}]}
)

In [0]:
# After invoking a LangGraph agent created with create_agent, the result contains a messages list with the full conversation history. The last message is the agent's final response.
final_response = result["messages"][-1].content
print(final_response)

In [0]:
for msg in result["messages"]:
    print(f"{type(msg).__name__}: {msg.content[:100]}...")
    print("---")

In [0]:
query = "Which documents do I need for the express entry application?"
result = agent.invoke(
    {"messages": [{"role": "user", "content": query}]}
)
final_response = result["messages"][-1].content
print(final_response)

In [0]:
query = "How high should I score for language tests for English if I apply under the CEC program? what is the IELTS score requirement? is there a tool to convert CLB-IELTS?"
result = agent.invoke(
    {"messages": [{"role": "user", "content": query}]}
)
final_response = result["messages"][-1].content
print(final_response)

## Wrapping with `ResponsesAgent` Interface

In [0]:
from typing import Generator
from langgraph.graph.state import CompiledStateGraph
from mlflow.models import set_model
from mlflow.pyfunc import ResponsesAgent
from mlflow.types.responses import (
    ResponsesAgentRequest,
    ResponsesAgentResponse,
    ResponsesAgentStreamEvent,
    output_to_responses_items_stream,
    to_chat_completions_input,
)


class LangGraphResponsesAgent(ResponsesAgent):
    def __init__(self, agent: CompiledStateGraph):
        self.agent = agent

    def predict(self, request: ResponsesAgentRequest) -> ResponsesAgentResponse:
        outputs = [
            event.item
            for event in self.predict_stream(request)
            if event.type == "response.output_item.done"
        ]
        return ResponsesAgentResponse(output=outputs, custom_outputs=request.custom_inputs)

    def predict_stream(
        self,
        request: ResponsesAgentRequest,
    ) -> Generator[ResponsesAgentStreamEvent, None, None]:
        cc_msgs = to_chat_completions_input([i.model_dump() for i in request.input])

        for _, events in self.agent.stream({"messages": cc_msgs}, stream_mode=["updates"]):
            for node_data in events.values():
                yield from output_to_responses_items_stream(node_data["messages"])


mlflow.langchain.autolog()
# graph = None  
graph = agent # TODO: replace with your compiled LangGraph agent
wrapped_agent = LangGraphResponsesAgent(graph)
# set_model(agent)

In [0]:
# test invocation:
wrapped_agent.predict(ResponsesAgentRequest(input=[{"role": "user", "content": "What is the minimum score for IELTS?"}]))

## Wrapping with `AgentServer`

In [0]:
from typing import AsyncGenerator, Optional
from databricks.sdk import WorkspaceClient
from mlflow.genai.agent_server import invoke, stream
from langchain.agents import create_agent
from langchain_core.tools import tool
from mlflow.types.responses import (
    ResponsesAgentRequest,
    ResponsesAgentResponse,
    ResponsesAgentStreamEvent,
    to_chat_completions_input,
)


sp_workspace_client = WorkspaceClient()
# TODO: populate the tools here

async def init_agent(workspace_client: Optional[WorkspaceClient] = None):
    agent = create_agent(
        model=llm_client, 
        tools=tools, 
        system_prompt=SYSTEM_PROMPT)
    return agent

@invoke()
async def invoke_handler(request: ResponsesAgentRequest) -> ResponsesAgentResponse:
    outputs = [
        event.item
        async for event in stream_handler(request)
        if event.type == "response.output_item.done"
    ]
    return ResponsesAgentResponse(output=outputs)

@stream()
async def stream_handler(
    request: ResponsesAgentRequest,
) -> AsyncGenerator[ResponsesAgentStreamEvent, None]:
    if session_id := get_session_id(request):
        mlflow.update_current_trace(metadata={"mlflow.trace.session": session_id})

    # By default, uses service principal credentials.
    # For on-behalf-of user authentication, use get_user_workspace_client() instead:
    #   agent = await init_agent(workspace_client=get_user_workspace_client())
    agent = await init_agent()
    messages = {"messages": to_chat_completions_input([i.model_dump() for i in request.input])}

    async for event in process_agent_astream_events(
        agent.astream(input=messages, stream_mode=["updates", "messages"])
    ):
        yield event